# Day 22: Important Notes
### Gradient, backpropagation, and what Adam remembers

Short notes for the questions we get in class:
1. What is a **gradient**?
2. How is it **calculated**? (backpropagation)
3. Why can the loss go **up** during training?
4. What does **Adam** store? Why is it different from SGD?
5. The **bias**: one per feature or one per neuron?

We use one tiny example all the time:

```
guess = w × 3        we want the guess to be 12  →  the best w is 4
loss  = (guess − 12)²                            we start at w = 2
```

In [ ]:
import torch

# 1. What is a gradient?

> **The gradient = how much the loss changes if I move this weight a tiny bit.**

| Gradient | Meaning | What to do with the weight |
|---|---|---|
| **negative** | moving the weight up makes the loss smaller | make it **bigger** |
| **positive** | moving the weight up makes the loss bigger | make it **smaller** |
| **big** (like 36) | the hill is steep: we are far from the bottom | |
| **near 0** | almost flat: we are near the bottom | |

**Like a shower knob with your eyes closed:** you turn it a tiny bit to the right and the water gets better. That feeling ("a tiny turn this way makes it better, by this much") is the gradient.

# 2. How is the gradient calculated?

### The slow way: try it

Move w a tiny bit (from 2 to 2.01) and see how much the loss changes.

In [ ]:
def loss_at(w):
    guess = w * 3
    return (guess - 12) ** 2

before = loss_at(2.00)                          # loss at w = 2
after = loss_at(2.01)                           # loss after a tiny move up
print("loss at 2.00:", before)
print("loss at 2.01:", round(after, 2))
print("gradient ≈", round((after - before) / 0.01))   # change in loss ÷ change in w

The loss went **down by 0.36** when w went **up by 0.01** → gradient ≈ **−36**.

This works, but a real network has **millions** of weights. Trying each one would take forever.

### The fast way: the chain rule (multiply the "reactions")

The loss is made in 2 steps. We ask how strongly each step reacts, and **multiply**:

```
w  ──(× 3)──▶  guess  ──(error²)──▶  loss
```

| Link | How strongly does it react? | At w = 2 |
|---|---|---|
| w → guess | guess = w × 3: the guess moves **3 times** as much as w | **3** |
| guess → loss | loss = error²: it reacts with **2 × error** | 2 × (6 − 12) = **−12** |
| **w → loss** | **multiply the links** | 3 × (−12) = **−36** ✓ |

**Like gears:** if gear A turns gear B 3 times faster, and gear B turns gear C −12 times faster, then A turns C 3 × (−12) = −36 times faster.

### Backpropagation = the chain rule, from the end back to the start

```
loss  ◀──  output layer  ◀──  hidden layer  ◀──  inputs
      (multiply the links, going backwards, layer by layer)
```

- `requires_grad=True` → during the forward pass, PyTorch **remembers every calculation**.
- `loss.backward()` → it walks **back** through them, multiplies the links, and writes each gradient into `.grad`.
- One trip back gives the gradient of **every** weight. That's why it is fast.

In [ ]:
w = torch.tensor(2.0, requires_grad=True)       # our weight. Track it!
loss = (w * 3 - 12) ** 2                        # forward pass + loss: (6 − 12)² = 36
loss.backward()                                 # backpropagation: the chain rule, done for us
print("loss:", loss.item())
print("gradient of w:", w.grad.item())          # −36, the same as our hand calculation

# 3. Why can the loss go UP during training?

The optimizer **does not look at the old loss**. It never checks "did my step make things better?".
It only uses the **gradient**: the slope **where it stands right now**. Then it steps.

The gradient tells **which way** is down, but **not where the bottom is**. If the step is too big, it **jumps over the bottom** and lands higher on the other side.

**new w = w − learning rate × gradient**. Start at w = 2 (loss 36, gradient −36):

In [ ]:
for lr in [0.01, 0.05, 0.1, 0.2]:
    new_w = 2 - lr * (-36)                      # one step of gradient descent
    print(f"lr = {lr:<5}  new w = {new_w:4.2f}   new loss = {loss_at(new_w):6.2f}")

| Learning rate | What happened |
|---|---|
| 0.01 | small step, loss goes down ✓ |
| 0.05 | good step, almost at the bottom (w ≈ 4) ✓ |
| 0.1 | jumped **over** 4, but still lower than 36 |
| 0.2 | jumped far over 4: the loss went **UP** from 36 to 243 ✗ |

**Then why does it come back?** At w = 9.2 the gradient is now **positive** ("make w smaller"), so the next step goes back. If the steps are not too big, each swing gets smaller and it settles at the bottom.
That's the **bump** you saw in the house-price loss curve: most houses got worse **together** for a few epochs, then it calmed down.

**Why not check and undo bad steps?** It would mean calculating the loss again after every step. With millions of weights that is slow, and the bumps fix themselves. So we just choose a learning rate that is not too big.

# 4. What does Adam store?

**SGD stores nothing.** Every step: `w = w − lr × gradient`. The step size depends on the gradient size.

**Adam stores 3 things for every weight:**

| Name in PyTorch | In simple words | Why |
|---|---|---|
| `step` | how many steps so far | a small correction at the start |
| `exp_avg` (**m**) | the **average direction** of recent gradients | **momentum**: "I've been going up for a while, keep going" (a rolling ball) |
| `exp_avg_sq` (**v**) | the **average size** of recent gradients (squared) | "how steep is it usually, **for this weight**?" |

**How Adam steps:**

```
step = lr × (average direction m) ÷ (usual size √v)
```

Dividing by the usual size makes every weight step **about the same amount (≈ lr)**:
- a weight with **huge** gradients → divided by a big number → a careful step
- a weight with **tiny** gradients → divided by a small number → still a useful step

Let's look inside Adam for our weight:

In [ ]:
w = torch.tensor(2.0, requires_grad=True)
optimizer = torch.optim.Adam([w], lr=0.1)

for step in range(1, 4):
    loss = (w * 3 - 12) ** 2
    optimizer.zero_grad()
    loss.backward()
    gradient = w.grad.item()
    optimizer.step()

    memory = optimizer.state[w]                 # what Adam remembers for this weight
    print(f"step {step}: gradient = {gradient:6.1f}   "
          f"m = {memory['exp_avg'].item():6.2f}   "
          f"v = {memory['exp_avg_sq'].item():5.2f}   new w = {w.item():.2f}")

print("Adam stores:", list(optimizer.state[w].keys()))

**What we see:**
- Each step moves w by about **0.1** (= lr), even though the gradient is huge (−36).
- **m** grows step by step: Adam builds up **speed** in the same direction.

**Compare with SGD, lr = 0.1:** w = 2 − 0.1 × (−36) = **5.6**: one giant jump past 4, because SGD uses the raw gradient size.

**This is why Adam worked on our students without scaling:** the study and phone weights had very different gradients, and Adam balanced their step sizes by itself.

# 5. The bias: one per feature, or one per neuron?

**One bias per neuron.** Each neuron has **one weight for every feature**, and **one bias**, added once at the end.

| Neuron | weight for study | weight for phone | bias |
|---|---|---|---|
| 1 | 0.5 | −0.6 | **0.1** |
| 2 | −1.0 | 0.4 | **0.2** |
| 3 | 0.8 | 0.2 | **−0.5** |

- In one epoch, **every student** uses the **same** weights and the **same** bias. Only the inputs change.
- After `optimizer.step()`, the weights **and** the biases change a little (they are learned too).
- After training, they stay **fixed**.

`nn.Linear(2, 3)` → 2 × 3 = **6 weights** + **3 biases**.

In [ ]:
from torch import nn

layer = nn.Linear(2, 3)
print("weights:", layer.weight.shape)           # (3, 2): 3 neurons, 2 weights each (one per feature)
print("biases: ", layer.bias.shape)             # (3,): one bias per neuron

# Summary

- **Gradient:** if I move this weight a tiny bit, how much does the loss change? The sign says which way, the size says how steep.
- **Backpropagation** (`loss.backward()`): the chain rule, going backwards. It multiplies the "reactions" and finds the gradient of **every** weight in one trip.
- **The loss can go up**: the optimizer only feels the slope, it never checks the old loss. A step that is too big jumps over the bottom.
- **SGD:** step = lr × gradient. Remembers nothing.
- **Adam:** remembers the **direction** (m) and the **usual size** (v) of each weight's gradients. Every weight steps about the same amount, and it speeds up in a steady direction.
- **Bias:** one per neuron, the same for every student in an epoch, learned like the weights.